# How To Use

Following the datasets used in the 'Vulnerability to One-Pixel Attacks of Neural Network Architectures in Medical Image Classification' paper, we are using two different datasets:
 - https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset
   - The training set from this will be our TRAIN set
   - The test set from this will be our VALIDATION set
 - https://www.kaggle.com/datasets/sartajbhuvaji/brain-tumor-classification-mri/versions/2?select=Training
   - This ENTIRE dataset will be our TEST set

For this to work you need to download both of these, make and make sure the folder names are correct.
- Rename the 'Testing' folder from this dataset https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset to 'Validation'
- Put both folders of this dataset https://www.kaggle.com/datasets/sartajbhuvaji/brain-tumor-classification-mri/versions/2?select=Training into a folder called 'Testing'

You should get an output like:

```
Manifest created with 10464 images.
label  glioma  meningioma  notumor  pituitary
split                                        
test      926         937      500        901
train    1400        1400     1400       1400
val       400         400      400        400
```

In [1]:
import os
import pandas as pd

def generate_csv(root_dirs, output_csv="../data/dataset.csv"):
    data = []
    
    # The different datasets used different label names
    label_map = {
        'pituitary': 'pituitary',
        'pituitary_tumor': 'pituitary',
        'notumor': 'notumor',
        'no_tumor': 'notumor',
        'meningioma': 'meningioma',
        'meningioma_tumor': 'meningioma',
        'glioma': 'glioma',
        'glioma_tumor': 'glioma'
    }

    for split_name, root_path in root_dirs.items():
        for root, dirs, files in os.walk(root_path):
            for file in files:
                if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                    # The parent folder is the label
                    folder_label = os.path.basename(root)
                    
                    if folder_label in label_map:
                        clean_label = label_map[folder_label]
                        full_path = os.path.join(root, file)
                        
                        data.append({
                            "abs_path": os.path.abspath(full_path),
                            "label": clean_label,
                            "split": split_name
                        })
    
    df = pd.DataFrame(data)
    df.to_csv(output_csv, index=False)
    print(f"Manifest created with {len(df)} images.")
    print(df.groupby(['split', 'label']).size().unstack(fill_value=0))

# Define your specific folder structure
dirs = {
    'train': '../data/Training',
    'val': '../data/Validation',
    'test': '../data/Testing'
}

generate_csv(dirs)

Manifest created with 10464 images.
label  glioma  meningioma  notumor  pituitary
split                                        
test      926         937      500        901
train    1400        1400     1400       1400
val       400         400      400        400


In [2]:
df = pd.read_csv('../data/dataset.csv')
train_paths = set(df[df['split'] == 'train']['abs_path'])
test_paths  = set(df[df['split'] == 'test']['abs_path'])
overlap = train_paths & test_paths
print(f"Overlapping images: {len(overlap)}")

Overlapping images: 0


In [3]:
print(df[df['split'] == 'test']['abs_path'].str.contains('Testing/Training').sum(), "from Testing/Training")
print(df[df['split'] == 'test']['abs_path'].str.contains('Testing/Testing').sum(), "from Testing/Testing")


2870 from Testing/Training
394 from Testing/Testing
